In [1]:
import re
import json
import nltk
import pandas as pd

from typing import Dict, List, Set, Tuple, Optional, TypedDict, Union
from pandas._libs.missing import NAType
from nltk.corpus import stopwords
from pathlib import Path

In [2]:
nltk.download('stopwords')
STOP_WORDS: Set[str] = set(stopwords.words('spanish'))

def clean_concept(concept: str):
    if concept == "" or concept is None:
        return ""
    if (not isinstance(concept, str)):
        return str(concept)

    concept = concept.lower()
    tabla = str.maketrans(f"áäéëíïóöúü", "aaeeiioouu")
    concept = concept.translate(tabla)
    concept = re.sub(r'\(\s*[\d,.]+\s*%?\s*\)', '', concept)
    concept = re.sub(r'\b\d+\s*[xX]\s*\d+\b', '', concept)
    concept = re.sub(r'\b\d+(?:[.,]\d+)?\s*%', '', concept)
    meses: list[str] = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]
    pattern_meses = r'\b(?:' + '|'.join(meses) + r')\b'
    concept = re.sub(pattern_meses, 'mes', concept)
    concept = re.sub(r'\b\d+(?:[.,]\d+)?\b', '', concept)
    concept = re.sub(r'\s*[,.]\s*', ' ', concept)
    caracteres_especiales: str = '–-#()[]{}/:_*+.,~°";$&´='+"'"
    tabla = str.maketrans(caracteres_especiales, " " * len(caracteres_especiales))
    concept = concept.translate(tabla)
    concept = re.sub(r'\d+$', '', concept)
    concept = re.sub(r'[0-9]', ' ', concept)
    concept = re.sub(r'\s+', ' ', concept).strip()
    tokens = concept.split()
    tokens = [word for word in tokens if word not in STOP_WORDS and len(word) > 2]
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_CONCEPTOS_PTESA: str = "./resources/input/Conceptos PTESA.xlsx"

df_conceptos: pd.DataFrame = pd.read_excel(INPUT_CONCEPTOS_PTESA)
df_conceptos: pd.DataFrame = df_conceptos['Concepto'].astype(str).str.split('|').explode().to_frame()
df_conceptos["concepto_cleared"] = df_conceptos["Concepto"].apply(clean_concept)
df_conceptos.head()

,Concepto,concepto_cleared
0,COSTO DIRECTO DE OBRA,costo directo obra
0,ADMINISTRACION (22%),administracion
0,IMPROVISTOS (3%),improvistos
0,"UTILIDAD (4,2016806722%) MAS IVA 19%",utilidad mas iva
0,ESTUDIOS Y DISEÑOS MAS IVA 19%,estudios diseños mas iva


In [4]:
INPUT_HOMOLOGA: str = "./resources/input/Tabla_homologación_2.xlsx"

def get_homologa_dict() -> Dict[str, str]:
    df_homologa: pd.DataFrame = pd.read_excel(INPUT_HOMOLOGA)
    df_keywords: pd.DataFrame = df_homologa[~df_homologa["KeyWords"].isna()].copy()
    dict_keyword_concept: Dict[str, str] = {}
    for ix, row in df_keywords.iterrows():
        cleared_keywords: List[str] = [clean_concept(kw) for kw in row["KeyWords"].split(",") if len(clean_concept(kw)) > 0]
        for kw in cleared_keywords:
            dict_keyword_concept[kw] = row["Concepto"]
    return dict_keyword_concept

homologa_dict: Dict[str, str] = get_homologa_dict()

In [5]:
class ClasificacionItem(TypedDict):
    concepto: str    # Concepto original (espejo del input para verificar integridad)
    categoria: str   # Uno de los valores de CONCEPTOS_TRIBUTARIOS
    explicacion: str # Justificación breve (máx ~40 palabras)

BACKUP_DIR: Path = Path("./resources/output/backup_llm_answer")
sessions: List[Path] = [d for d in BACKUP_DIR.iterdir() if d.is_dir()]
dict_conceptos: Dict[str, ClasificacionItem] = {}
cantidad_conceptos: int = 0
missing_concepts: List[str] = []
for folder in sessions:
    files: List = [f for f in folder.iterdir() if f.is_file()]

    #flag: bool = False
    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            document: Dict[str, List] = json.loads(f.read())
            input: List[str] = document["input"]
            output: Dict[str, ClasificacionItem] = {x["concepto"]:x for x in document["output_raw"]}
            cantidad_conceptos += len(input)
            for concepto in input:
                if concepto not in output:
                    #print(f"Concepto [{concepto}] not está en el dict: {output.keys()}")
                    #flag: bool = True
                    missing_concepts.append(concepto)
            dict_conceptos: Dict[str, ClasificacionItem]  = output | dict_conceptos
        #if flag:
            #break

df_llm_answers: pd.DataFrame = pd.DataFrame(list(dict_conceptos.values()))
df_llm_answers: pd.DataFrame = df_llm_answers[['concepto', 'categoria', 'explicacion']]
print("cantidad de conceptos obtenidos:", len(df_llm_answers))
print("Cantidad de conceptos procesados:", cantidad_conceptos)
print("Cantidad de conceptos faltantes:", len(missing_concepts), f"| Porcentaje faltante: {100*len(missing_concepts)/cantidad_conceptos:.2f}%")
print("Cantidad de conceptos PTESA:", len(df_conceptos["concepto_cleared"].drop_duplicates()))
df_llm_answers.head()

cantidad de conceptos obtenidos: 22843
Cantidad de conceptos procesados: 22892
Cantidad de conceptos faltantes: 47 | Porcentaje faltante: 0.21%
Cantidad de conceptos PTESA: 23115


,concepto,categoria,explicacion
0,nan,DESCONOCIDO,"Valor nulo, sin contenido informativo para rea..."
1,,DESCONOCIDO,El concepto no contiene información válida par...
2,dulceabrigo rojo metro,Compras Generales,Bien corporal (prenda de vestir) considerado c...
3,servicio alquiler servidor,Arrendamiento de bienes muebles,Alquiler de equipos de computo y servidores qu...
4,asesoria apoyo direccion estrategica ene,Consultoría General/Administración Delegada (PJ),Servicio de asesoria y consultoria estrategica...


In [6]:
def clasifly_using_llm_answer(concept: str) -> Tuple[Union[str, NAType], Union[str, NAType], Union[str, NAType]]:
    if not isinstance(concept, str) or len(concept) <= 3:
        return ("CONCEPTO VACIO", "NO-DATA", pd.NA)
    for kw, cat in homologa_dict.items():
        if kw == concept or kw in concept:
            return (cat, "KEYWORD", pd.NA)
    if concept in dict_conceptos:
        return (dict_conceptos[concept]["categoria"], "gemini-3.1-flash-lite-preview", dict_conceptos[concept]["explicacion"])
    return (pd.NA, pd.NA, pd.NA)

print("Procesando LLM answers")
df_conceptos["categoria"] = pd.NA
df_conceptos["alg"] = pd.NA
df_conceptos["explicacion"] = pd.NA
df_conceptos[["categoria", "alg", "explicacion"]] = df_conceptos["concepto_cleared"].apply(lambda x: pd.Series(clasifly_using_llm_answer(x)))
df_conceptos.head()

Procesando LLM answers


,Concepto,concepto_cleared,categoria,alg,explicacion
0,COSTO DIRECTO DE OBRA,costo directo obra,Contratos de construcción y urbanización,KEYWORD,NaN
0,ADMINISTRACION (22%),administracion,Servicios Generales,gemini-3.1-flash-lite-preview,Concepto genérico sin descripción suficiente d...
0,IMPROVISTOS (3%),improvistos,Servicios Generales,gemini-3.1-flash-lite-preview,Concepto contable genérico sin actividad defin...
0,"UTILIDAD (4,2016806722%) MAS IVA 19%",utilidad mas iva,Servicios Generales,gemini-3.1-flash-lite-preview,Componente de un contrato sin detalle de la ac...
0,ESTUDIOS Y DISEÑOS MAS IVA 19%,estudios diseños mas iva,Consultoría General/Administración Delegada (PJ),gemini-3.1-flash-lite-preview,Los estudios y diseños corresponden a labores ...


In [7]:
df_conceptos.to_excel("./resources/output/result_llm_2.xlsx", index=False)

In [8]:
df_checkpoint: pd.DataFrame = df_conceptos[["concepto_cleared"]].drop_duplicates()
df_checkpoint[["categoria", "metodo_algoritmo", "explicacion"]] = df_checkpoint["concepto_cleared"].apply(lambda x: pd.Series(clasifly_using_llm_answer(x)))
df_checkpoint.to_csv("./resources/output/fast_checkpoint.csv", index=False)

In [9]:
total: int = len(df_conceptos)
faltante: int = len(df_conceptos[df_conceptos["categoria"].isna()])

print("Cantidad total:", total)
print("Cantidad faltante:", faltante)
print(f"Porcentaje faltante: {100*faltante/total:.2f}%")
print(f"Conceptos faltantes:", len(df_conceptos[df_conceptos["categoria"].isna()]["concepto_cleared"].drop_duplicates()))
df_conceptos[df_conceptos["categoria"].isna()]["concepto_cleared"].drop_duplicates().head(30)

Cantidad total: 131337
Cantidad faltante: 0
Porcentaje faltante: 0.00%
Conceptos faltantes: 0


Series([], Name: concepto_cleared, dtype: str)